# promote

Carries Saved tables from your Dev Database to your Prod Database.

## Terms

- **Step**: each numbered heading below, with the code cell under it. Its first line says its kind: it changes only files, or only prints; it is a call to the warehouse, yours to run; or it is a decision, yours to make.
- **Names cell**: the code cell after the one that finds the Project. It holds this Workflow's inputs, the start materials you fill in once, each with what to write in its comment; each step's call reads them.
- **Decisions cell**: `DECISIONS`, the code cell after the names cell, where you answer each open Decision a step waits on, by the table's name.
- **Saved table**: a table in the warehouse that only your own code writes, its files in a folder of its own.
- **Dev Database**: the database where you create and write tables of your own, each built, written and compared there first: `DEV` in admin/databases.py.
- **Prod Database**: the database others read your finished Saved tables from, which a table reaches only once promoted from the Dev Database: `PROD` in admin/databases.py.
- **Decision**: one thing about one table that only you decide, such as its location or its Key; the table's files say whether it is answered yet.
- **Table reference**: a script in references/ describing one table: its columns and their types, its Partitions and its Key.
- **Table folder**: the folder in tables/ that holds everything about one Saved table: its Table notebook, its past lineage, and its Hive.
- **Reads itself**: a table whose own script reads it, such as a running total, seeded once from its original before its first write.
- **Original**: the table a converted script writes, in the warehouse already, which its Saved table copies; for a query alone, such as one Copilot wrote from a workbook, the rows the query gives.
- **Session setting**: a `SET` line sent before a Statement, which it needs to run.
- **Partition**: the rows of a table that share one value of each column it is partitioned by.
- **Reference sheet**: references/reference_sheet.csv, one table listing every column of every table the Project has a Table reference for.
- **Database module**: a file such as references/ops.py, which gives each table of one database by name, such as ops.jobs.
- **Key**: the columns whose values name exactly one row of a table.
- **Table notebook**: the notebook in a Table folder whose cells create, preview and write its Saved table; its logic cell is the one whose first line is `# logic`.
- **Hive**: the SQL your warehouse runs.
- **Statement**: a query your code builds, which becomes one Hive string.

## Before you start

Fill in the names cell below and save the notebook: each call reads it, and so does `project.check()`, which names the next step.

## Decision rules

- If you want the steps run for you, as most do: fill in the names cell and save the notebook, run "Run up to your Decisions", paste what it prints into the decisions cell under it, fill it in and save, then run "Run the rest". Each runs every step left in turn, up to the first only you can do, and says where it stopped.
- If you run the steps yourself, run each step's cell under "Step by step", in turn, in order.
- If you want only the steps that change files run for you, run `project.do_next()`, the last cell. It runs them up to the first it can't.

## Self-check

Once the last step is done, each of these holds:

- Each dev table in `TABLES` has a prod copy: its Table folder in tables/ and its Table reference in references/.
- Each prod copy's Table reference has its `location=` and leaves no Decision open.
- admin/sent.csv records each prod copy created and written, and each that reads itself seeded.
- references/reference_sheet.csv lists each prod copy's columns.
- `project.check()` names no next step of this Workflow.

In [ ]:
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[2]))  # the Project: three up

from sqlglot_composer import admin, hive_settings, project, run_execute, run_query, show_hive
from warehouse import query, execute

In [ ]:
TABLES = []  # the dev tables, by their names in your Dev Database, such as ["mart_dev.daily_job_runs"]: left [], the run cells promote every finished table not promoted yet, or changed in dev since, naming each and why

## Run up to your Decisions

This cell and "Run the rest" are one call, `project.run_workflow`, run once before you answer and once after. Run the cell below: it runs every step left, up to the first only you can do, and says where it stopped. At a Decision, it prints a whole DECISIONS: paste it into the decisions cell under it, fill in each <FILL IN>, and save the notebook.

In [ ]:
# Run up to your Decisions: every step left, up to the first only you can do, such as a
# Decision, whose DECISIONS it prints to paste into the cell under it.
# Add dry_run=True to see the Hive it would send, sending nothing through your
# execute: help(project.run_workflow) says the steps it still runs:
project.run_workflow("promote", query=query, execute=execute)

In [ ]:
DECISIONS = {}  # your answer to each open Decision, by the table's name as
# project.check() writes it, such as
# {"mart_dev.weekly_ops": {"key": ["job_id"], "types": {"fails": "bigint"},
#                          "tags": {"PERIOD_END": "'2026-10-31'"}}}
# "key": None says no columns name one row. project.check() names each one open, and
# prints a whole DECISIONS to paste over this line: fill in each <FILL IN>. A Key
# convert or new_table proposed, marked # proposed, is kept unless you answer "key"
# here; one add_references proposed stays open until you accept it.
# "key": "proposed" accepts either as if typed. "all_tables" answers for every table
# of this notebook, and a table's own answer overwrites it, kind by kind and tag by
# tag:
# {"all_tables": {"key": "proposed",
#                 "tags": {"PERIOD_END": "'2026-10-31'"}}}
# accepts each table's proposed Key and gives each table holding PERIOD_END that
# value. A table's location is placed under "location_root" in
# admin/databases.py; answer "location" only for one it can't place.
# A tag may be answered with a Run value's function of your Project's, imported above
# DECISIONS, such as "PERIOD_END": period_end. For a table's latest Partition, write
# project.new_run_value("period_end", latest_partition="ops.job_runs.dt") at the top
# of this cell, the import it prints under it, and run the cell once: it writes the
# function into building_blocks/run_values.py. And "seed", such as
# {"dt": {"from": "2026-01-01"}}, limits the Partitions a table that reads itself
# copies: how-to 09 shows both.

## Run the rest

Run the cell below once the decisions cell above is filled in and saved: the same call again, it goes on from where the cell before stopped, and it too stops at the first step only you can do: a refusal, a compare that failed, or a Decision left open, each named. Mend what it names, such as an answer in the decisions cell, save, and run this cell again.

In [ ]:
# Run the rest: the same call, run again once you have answered: it goes
# on from where it stopped, and stops again at a refusal, a compare that
# failed or a Decision left open.
# Add dry_run=True to see the Hive it would send, sending nothing through your
# execute: help(project.run_workflow) says the steps it still runs:
project.run_workflow("promote", query=query, execute=execute)

# Step by step

Each step's own cell, to run or run again one step on purpose; the cells above run them all for you.

## 1. Promote each table: a prod copy of its folder and Table reference

It changes only files, so project.do_next() runs it for you.

- **Goal:** a prod copy of each dev table in `TABLES`, its own Table folder and Table reference, each reading the prod copies of the others promoted with it.
- **Action:** Run the cell below.
- **Done when** each dev table in `TABLES` has its prod copy's Table folder in tables/; then step 2.

In [ ]:
project.promote(TABLES)

## 2. Answer each prod table's open Decisions in the decisions cell

It is a decision, yours to make.

- **Goal:** an answer to each prod copy's open Decision: its location, the folder its files go in.
- **Action:** Run the cell below, `project.check()`: it names each open Decision and prints a whole `DECISIONS` to paste over the decisions cell's. If it names no open Decision of a prod copy, there is nothing to answer: step 1 filled in each prod copy's `location=` from its dev one, its folder named for your Prod Database, or from the one it had when promoted before. Leave the decisions cell as it is, and go on. Else write each answer in the decisions cell, `DECISIONS`, by the prod table's name as `project.check()` writes it, then save the notebook.
- **Done when** the decisions cell answers each Decision `project.check()` names for the prod copies, or it names none; then step 3.

In [ ]:
project.check()

## 3. Write each answer into its prod table's Table reference

It changes only files, so project.do_next() runs it for you.

- **Goal:** each answer written into its prod copy's Table reference, which then has its `location=`.
- **Action:** Run the cell below.
- **Done when** each prod copy's Table reference leaves no Decision open; then step 4.

In [ ]:
project.write_decisions(DECISIONS, tables=project.promoted_tables(TABLES))

## 4. Create each table in prod

It is a call to the warehouse, yours to run.

- **Goal:** each prod copy made in your Prod Database.
- **Action:** Run the cell below. It sends each prod copy's `create()` through your execute.
- **Done when** admin/sent.csv records a create of each prod copy; then step 5.

In [ ]:
for table in project.promoted_tables(TABLES):
    run_execute(project.load(table).create(), execute)

## 5. Seed each prod table that reads itself, once, from its original

It is a call to the warehouse, yours to run.

- **Goal:** each prod copy that reads itself holding its original's rows, once: a prod copy's original is its dev copy's original.
- **Action:** Run the cell below. It sends the Session setting their Partitions need, then each such prod copy's `seed()`, through your execute. If no prod copy reads itself, then this step is done once step 4 is, and the cell sends nothing and prints that it seeded nothing.
- **Done when** admin/sent.csv records the seed of each prod copy that reads itself; then step 6.

In [ ]:
to_seed = project.promoted_tables(TABLES, seeded=True)
if not to_seed:
    print("Seeded nothing: no prod copy reads itself.")
else:
    run_execute(hive_settings("dynamic_partitions"), execute)
    for table in to_seed:
        run_execute(project.load(table).seed(), execute)

## 6. Write each table in prod

It is a call to the warehouse, yours to run.

- **Goal:** each prod copy's rows written in your Prod Database.
- **Action:** Run the cell below. `project.run_tables` sends each prod copy's `write()` through your execute.
- **Done when** admin/sent.csv records each prod copy written since it was last promoted; then step 7.

In [ ]:
project.run_tables(tables=project.promoted_tables(TABLES), execute=execute)

## 7. Refresh the Reference sheet and the Database modules

It changes only files, so project.do_next() runs it for you.

- **Goal:** each prod copy in the Reference sheet and in its Database module.
- **Action:** Run the cell below.
- **Done when** references/reference_sheet.csv is written again after each prod copy's Table reference last changed; the Workflow is done: go through the Self-check at the top.

In [ ]:
project.refresh_reference_sheet(project.promoted_tables(TABLES))

In [ ]:
project.do_next()